# Lesson 04 - Pattern ng Disenyo ng Paggamit ng Kasangkapan

Sa araling ito, matututuhan mo ang **Tool Use** na pattern ng disenyo para sa AI agents gamit ang Microsoft Agent Framework (Python). Tatalakayin natin:

- Pagtukoy ng mga function tools gamit ang `@tool` decorator at may typed parameters
- Pagbibigay ng mga tool schemas upang malaman ng modelo kung ano ang ginagawa ng bawat kasangkapan
- Pagkontrol sa pagpapatupad ng kasangkapan gamit ang `approval_mode`
- Pagbibigay ng **istrakturang output** sa pamamagitan ng mga Pydantic models at `response_format`

Ang senaryo ay isang **travel booking agent** na maaaring maghanap ng mga destinasyon, mag-check ng availability, at kumuha ng impormasyon tungkol sa flight.


## Setup


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Pagde-define ng Mga Tool gamit ang @tool Decorator

Ang `@tool` decorator ay nagiging isang simpleng Python function na isang tool na maaaring tawagin ng isang ahente.
Mga pangunahing punto:

- Ang **docstring** ay nagiging deskripsyon ng tool na nakikita ng modelo.
- Ang **type annotations** (kabilang ang `Annotated` na may mga deskripsyon) ay nagtatakda ng schema ng tool.
- Ang `approval_mode` ay kumokontrol kung kailangang aprubahan ng user ang bawat tawag bago ito isagawa.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Paglikha ng Ahente na may Maramihang Mga Kasangkapan

Ipasa ang lahat ng tatlong kasangkapan sa kliyente upang magamit ng modelo ang alinman sa mga ito na kailangan nito upang sagutin ang tanong ng gumagamit.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Structured Output with Tools

Sa pamamagitan ng pagtatakda ng `response_format` sa isang Pydantic na modelo, pinipilit ang agent na magbalik ng isang mahusay na tayp na JSON object sa halip na malayang anyo ng teksto. Kapaki-pakinabang ito kapag kailangang gamitin ng downstream na code ang resulta nang programmatic.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Mga Pattern ng Pag-apruba ng Tool

Ang parameter na `approval_mode` sa `@tool` ay kumokontrol kung ang mga tawag sa tool ay nangangailangan ng pag-apruba ng tao bago isagawa:

| Mode | Pag-uugali |
|---|---|
| `"never_require"` | Awtomatikong tumatakbo ang tool — hindi kailangan ng kumpirmasyon mula sa user. |
| `"always_require"` | Bawat tawag ay kailangang aprubahan ng user bago ito isagawa. |

Gamitin ang `"always_require"` para sa mga tool na may mga epekto sa labas (hal. pag-book ng flight, pagsingil sa credit card) upang manatiling kasali ang tao sa proseso.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Buod

Sa araling ito natutunan mo kung paano:

1. **Magdeklara ng mga tool** gamit ang `@tool` na decorator na may typed parameters at docstrings na nagsisilbing schema ng tool.
2. **Pagsamahin ang maraming tool** upang ang ahente ay maaaring tawagan ang mga ito nang sunud-sunod para sagutin ang mga komplikadong tanong.
3. **Magbalik ng nakaayos na output** sa pamamagitan ng pagpapasa ng Pydantic model bilang `response_format`.
4. **Kontrolin ang pag-apruba ng tool** gamit ang `approval_mode` upang mapanatili ang tao sa proseso para sa mga sensitibong operasyon.

Ang mga pattern na ito ay bumubuo ng pundasyon para makalikha ng maaasahan at handang gamitin sa produksyon na mga ahente na maaaring makipag-ugnayan sa mga panlabas na sistema nang ligtas.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagsusuri**:  
Ang dokumentong ito ay isinalin gamit ang AI translation service na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat nagsusumikap kami para sa pagiging tumpak, mangyaring tandaan na maaaring may mga pagkakamali o hindi pagkakatugma sa awtomatikong pagsasalin. Ang orihinal na dokumento sa sariling wika nito ang dapat ituring na opisyal na sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na maaaring lumitaw mula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
